In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pickle
import numpy as np
import bloom


In [171]:
df = pd.read_csv("dataset/urldata.csv")
len(df)

420464

In [172]:
df['label'] = df['label'].map({'good':1,'bad':0})
good_urls = df[df['label']==1]['url'].tolist()
bad_urls = df[df['label']==0]['url'].tolist()

In [173]:
train_df,test_df = train_test_split(df,test_size=0.3,random_state=42)

In [204]:
vectorizer = HashingVectorizer(n_features=2**12,alternate_sign=False)
X_train = vectorizer.fit_transform(train_df['url'])
X_test = vectorizer.transform(test_df['url'])
y_train = train_df['label']
y_test = test_df['label']

In [205]:
model = LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [206]:
pickle.dump(model,open("model.pkl","wb"))
pickle.dump(vectorizer,open("vectorizer.pkl","wb"))

preparing url for standard bf

In [207]:
with open("good_urls.txt","w") as f:
    for url in good_urls:
        f.write(url + "\n")

In [208]:
with open("bad_urls.txt", "w", encoding="utf-8") as f:
    for url in df[df['label'] == 0]['url']:
        f.write(url + "\n")

preparing fn list for learned bf

In [209]:
probs = model.predict_proba(X_test)[:, 1]

In [210]:
threshold = 0.7
preds = (probs>=threshold).astype(int)

In [211]:
fn_mask = (y_test==1) & (preds==0)

In [212]:
fn_urls = test_df['url'][fn_mask].tolist()

In [213]:
print("Total positives:", sum(y_test == 1))
print("False negatives:", len(fn_urls))
print("FN rate:", len(fn_urls) / sum(y_test == 1))

Total positives: 103420
False negatives: 5171
FN rate: 0.05


In [214]:
false_positives = np.sum((y_test == 0) & (probs >= threshold))
total_negatives = np.sum(y_test == 0)

model_fpr = false_positives / total_negatives

print("Model FPR:", model_fpr)

Model FPR: 0.1809419014084507


In [215]:
with open("fn.txt", "w") as f:
    for url in fn_urls:
        f.write(url + "\n")

In [216]:
with open("test_urls.txt", "w", encoding= "utf-8") as fu, open("test_labels.txt", "w",encoding= "utf-8") as fl, open("test_probs.txt", "w",encoding= "utf-8") as fp:
    for url,label,prob in zip(test_df['url'],y_test,probs):
        fu.write(url+"\n")
        fl.write(f"{int(label)}\n")
        fp.write(f"{float(prob)}\n")

print("create test_urls,test_labels,test_probs")


create test_urls,test_labels,test_probs


evaculation loop

In [217]:
import os

model_size = os.path.getsize("model.pkl") / (1024 * 1024)
vectorizer_size = os.path.getsize("vectorizer.pkl") / (1024 * 1024)

print(model_size)
print(vectorizer_size)

0.0319366455078125
0.000396728515625


In [219]:
import subprocess
def run_cpp_eval():
    result = subprocess.run([".\eval"],capture_output=True,text=True)
    out = result.stdout.strip().splitlines()
    fn_count = int(out[0].split("=")[1])
    fpr = float(out[1].split("=")[1])
    size_mb = float(out[2].split("=")[1])
    total_neg = int(out[3].split("=")[1])
    total_pos = int(out[4].split("=")[1])
    return fn_count,fpr,size_mb


In [220]:
def run_cpp_standard():
    result  = subprocess.run([".\standard"],capture_output=True,text=True)
    out = result.stdout.strip().splitlines()
    stand_fpr = float(out[0].split("=")[1])
    stand_size = float(out[1].split("=")[1])
    stand_bits = float(out[2].split("=")[1])
    stand_querytime = float(out[3].split("=")[1])

    return stand_fpr,stand_size,stand_bits,stand_querytime

In [221]:
fn_count,fpr,size_mb = run_cpp_eval()
totalsize = size_mb+model_size+vectorizer_size
print(f"FN Count {fn_count} and FPR {fpr}")
print(f"backup bf {size_mb:.2f} in MB")
print(f"mobel size {model_size:.2f} in MB")
print(f"vectorizer size {vectorizer_size:.2f} in MB")
print(f" total size in MB {totalsize:.2f}")

FN Count 5171 and FPR 0.0768926
backup bf 0.01 in MB
mobel size 0.03 in MB
vectorizer size 0.00 in MB
 total size in MB 0.04


In [225]:
stand_fpr,stand_size,stand_bits,stand_querytime = run_cpp_standard()
print(f"Standard BF FPR {stand_fpr:.2f}")
print(f"Standard BF Size in MB {stand_size:.2f}")
print(f"Standard BF Bits/Element {stand_bits:.2f}")
print(f"Standard BF Query Time (nanoseconds) {stand_querytime:.2f}")

Standard BF FPR 0.07
Standard BF Size in MB 0.05
Standard BF Bits/Element 5.01
Standard BF Query Time (nanoseconds) 2144.97
